# 天気図パターン分類AI - Colabワークフロー

このノートブックは、リポジトリの`README.md`に沿ってColab上でデータ収集〜学習〜推論を行うための雛形です。
上から順にセルを実行してください。ランタイムが切れた場合は「セットアップ」のセルから再実行してください。

**GPU推奨**: メニュー → ランタイム → ランタイムのタイプを変更 → ハードウェアアクセラレータで「GPU(T4)」を選択

## 1. セットアップ(毎回のランタイムで最初に実行)

In [ ]:
REPO_URL = "https://github.com/awg-yk/weather-pattern-classification.git"
BRANCH = "claude/weather-chart-classification-4b6in1"
REPO_DIR = "/content/weather-pattern-classification"
DRIVE_DATA_DIR = "/content/drive/MyDrive/weather-pattern-classification-data"

import subprocess, os

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)

%cd {REPO_DIR}
!pip install -q -r requirements.txt
!apt-get -qq install -y poppler-utils fonts-noto-cjk

from google.colab import drive
drive.mount('/content/drive')

os.makedirs(f"{DRIVE_DATA_DIR}/processed", exist_ok=True)
os.makedirs(f"{DRIVE_DATA_DIR}/weights", exist_ok=True)
print("セットアップ完了")

## 2. データ収集(気象庁の天気図PDFをダウンロードしてPNG化)

日付範囲は必要に応じて変更してください。1日1回(0Z)分のダウンロードです。

In [ ]:
!python scripts/collect_jma.py --start 2025-05-01 --end 2026-04-30 --hours 0 --out data/raw/jma

### 2b. (任意)手動ダウンロードしたPDFの取り込み

気象庁JSMAPアーカイブは2022年10月1日以降のみ。それより古い天気図(2000年〜2022年9月ごろ)は
国立国会図書館デジタルコレクション(https://dl.ndl.go.jp/pid/12896309)から手動でダウンロードし、
このセルでまとめてPNGに変換できる。Google Driveにアップロードしたフォルダを指定するとよい。

In [ ]:
!python scripts/preprocess_jma.py --in-dir data/raw/jma/png --out-dir data/processed/jma
!cp -n data/processed/jma/*.png "{DRIVE_DATA_DIR}/processed/"

# 手動取り込み分(2b)があれば、それも前処理してDriveにコピーする
import os
if os.path.isdir("data/raw/ndl_manual/png") and os.listdir("data/raw/ndl_manual/png"):
    !python scripts/preprocess_jma.py --in-dir data/raw/ndl_manual/png --out-dir data/processed/ndl_manual
    !cp -n data/processed/ndl_manual/*.png "{DRIVE_DATA_DIR}/processed/"

!ls "{DRIVE_DATA_DIR}/processed" | wc -l

## 3. 前処理(余白クロップ・日時スタンプ消去)とDriveへの保存

In [ ]:
!python scripts/preprocess_jma.py --in-dir data/raw/jma/png --out-dir data/processed/jma
!cp -n data/processed/jma/*.png "{DRIVE_DATA_DIR}/processed/"
!ls "{DRIVE_DATA_DIR}/processed" | wc -l

## 4. ラベリング(チェックボックスで複数選択)

既にラベル済みの画像は自動でスキップされる。中断・再開が自由。

In [ ]:
import sys
sys.path.append(REPO_DIR)
from scripts.label_tool import run_labeling_session

run_labeling_session(
    images_dir=f"{DRIVE_DATA_DIR}/processed",
    labels_csv=f"{DRIVE_DATA_DIR}/labels.csv",
)

## 5. 学習

In [ ]:
!python -m src.train \
    --data-dir "{DRIVE_DATA_DIR}/processed" \
    --labels "{DRIVE_DATA_DIR}/labels.csv" \
    --epochs 30 \
    --batch-size 16 \
    --out "{DRIVE_DATA_DIR}/weights/model.pt"

## 6. 評価

In [ ]:
!python -m src.evaluate \
    --data-dir "{DRIVE_DATA_DIR}/processed" \
    --labels "{DRIVE_DATA_DIR}/labels.csv" \
    --weights "{DRIVE_DATA_DIR}/weights/model.pt"

## 7. Grad-CAMで判断根拠を可視化

手元の天気図画像をアップロードして、モデルがどこに注目して予測したか確認する。

In [ ]:
from scripts.gradcam import show_gradcam
from google.colab import files

uploaded = files.upload()

show_gradcam(
    image_path=list(uploaded.keys())[0],
    weights_path=f"{DRIVE_DATA_DIR}/weights/model.pt",
    top_k=3,
    apply_preprocess=True,
)

## 8. Web推論デモを起動

実行後、埋め込みページで画像をアップロードして分類を試せる。

In [ ]:
import os, time, subprocess

!pkill -9 -f uvicorn
time.sleep(2)

os.environ["MODEL_WEIGHTS"] = f"{DRIVE_DATA_DIR}/weights/model.pt"

proc = subprocess.Popen(
    ["uvicorn", "webapp.backend.main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=REPO_DIR,
    env=os.environ.copy(),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
time.sleep(5)
print("起動中か:", proc.poll() is None)
!curl -s http://localhost:8000/health

from google.colab import output
output.serve_kernel_port_as_iframe(8000, height=700)